# PACO original vs TESISPRIMERBATCH (Colab)

Este notebook ejecuta `FastPACO` del repositorio **original** con los mismos datasets y parametros de `TESISPRIMERBATCH` para comparar robustez cientifica.

In [ ]:
# Configuracion (igualar TESISPRIMERBATCH)
DATASET_ROOT = "/content/drive/MyDrive/subchallenge1"
DATASETS = ["lmircam_1", "lmircam_2"]

BENCHMARK_PIXELS = 2500
INNER_RADIUS = 8
OUTER_RADIUS = 55
CROP_HALF_SIZE = 100  # 200x200
CPU = 1

# Ruta al repo original en Colab.
# Opcion A: clonar desde GitHub (si tienes URL publica/privada accesible)
PACO_ORIGINAL_GIT_URL = ""

# Opcion B: carpeta local ya disponible en Colab
PACO_ORIGINAL_DIR = "/content/PACO-ORIGINAL/paco_original"

# Referencia observada en TESISPRIMERBATCH (ajustable)
REFERENCE_BATCH = {
    "lmircam_1": {"snr_max": 18.18, "snr_mean": 0.08, "pixels": 2500},
    "lmircam_2": {"snr_max": 16.72, "snr_mean": 0.03, "pixels": 2500},
}

SNR_MAX_TOL = 1.0
SNR_MEAN_TOL = 0.2


In [ ]:
# Setup
import os
import sys
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "astropy", "pandas", "matplotlib"])

# Montar Drive si se usa DATASET_ROOT en Drive
if DATASET_ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Clonar repo original si se entrego URL
if PACO_ORIGINAL_GIT_URL.strip() and not Path(PACO_ORIGINAL_DIR).exists():
    dst = Path(PACO_ORIGINAL_DIR).parent
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", PACO_ORIGINAL_GIT_URL, str(dst)])

if not Path(PACO_ORIGINAL_DIR).exists():
    raise RuntimeError(f"No existe PACO_ORIGINAL_DIR: {PACO_ORIGINAL_DIR}")

sys.path.insert(0, str(Path(PACO_ORIGINAL_DIR)))
from paco.processing.fastpaco import FastPACO

print("PACO original path:", PACO_ORIGINAL_DIR)
print("Dataset root:", DATASET_ROOT)


In [ ]:
def parse_dataset_name(name):
    parts = name.split('_')
    return '_'.join(parts[:-1]), int(parts[-1])


def load_dataset(base_dir, instrument, dataset_id):
    base = Path(base_dir)
    cube = fits.getdata(base / f"{instrument}_cube_{dataset_id}.fits")
    angles = fits.getdata(base / f"{instrument}_pa_{dataset_id}.fits").flatten()
    psf = fits.getdata(base / f"{instrument}_psf_{dataset_id}.fits")
    pixscale = float(fits.getdata(base / f"{instrument}_pxscale_{dataset_id}.fits").flatten()[0])
    return cube, angles, psf, pixscale


def crop_cube_center(cube, half_size=100):
    if cube.shape[1] <= (2 * half_size) or cube.shape[2] <= (2 * half_size):
        return cube
    cy, cx = cube.shape[1] // 2, cube.shape[2] // 2
    return cube[:, cy-half_size:cy+half_size, cx-half_size:cx+half_size]


def build_phi0_annulus(img_shape, n_pixels_target=2500, inner_radius=8, outer_radius=55):
    h, w = int(img_shape[0]), int(img_shape[1])
    cy, cx = h // 2, w // 2
    outer_eff = min(outer_radius, min(cy, cx) - 5)

    yy, xx = np.indices((h, w))
    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    mask = (rr >= inner_radius) & (rr <= outer_eff)
    coords = np.column_stack((xx[mask], yy[mask]))
    if len(coords) == 0:
        raise RuntimeError("No se pudieron generar phi0s validos")

    n_sel = min(int(n_pixels_target), len(coords))
    idx = np.linspace(0, len(coords) - 1, n_sel, dtype=int)
    return coords[idx].astype(np.int32)


def run_fastpaco_original(cube, angles, psf, pixscale, phi0s, cpu=1):
    # En PACO original, psf_rad esta en arcsec y se convierte internamente a pixeles con px_scale
    psf_rad_arcsec = 4.0 * pixscale

    model = FastPACO(
        image_stack=cube,
        angles=angles,
        psf=psf,
        psf_rad=psf_rad_arcsec,
        px_scale=pixscale,
    )

    t0 = time.perf_counter()
    a, b = model.PACOCalc(phi0s, cpu=cpu)
    elapsed = time.perf_counter() - t0

    with np.errstate(divide='ignore', invalid='ignore'):
        snr = np.divide(b, np.sqrt(a), out=np.zeros_like(b), where=(a > 0))
        snr = np.nan_to_num(snr, nan=0.0, posinf=0.0, neginf=0.0)

    return {
        "time_s": float(elapsed),
        "n_pixels": int(len(phi0s)),
        "snr": snr,
        "snr_max": float(np.nanmax(snr)),
        "snr_mean": float(np.nanmean(snr)),
        "snr_std": float(np.nanstd(snr)),
        "count_snr_gt_3": int(np.sum(snr > 3.0)),
        "count_snr_gt_5": int(np.sum(snr > 5.0)),
        "count_snr_gt_7": int(np.sum(snr > 7.0)),
    }


In [ ]:
# Ejecutar PACO original en los mismos datasets del primer batch
results_original = {}
rows = []

for ds_name in DATASETS:
    instrument, dataset_id = parse_dataset_name(ds_name)
    print("\n" + "="*70)
    print(f"Procesando con PACO original: {ds_name}")
    print("="*70)

    cube, angles, psf, pixscale = load_dataset(DATASET_ROOT, instrument, dataset_id)
    cube = crop_cube_center(cube, half_size=CROP_HALF_SIZE)
    phi0s = build_phi0_annulus(
        img_shape=(cube.shape[1], cube.shape[2]),
        n_pixels_target=BENCHMARK_PIXELS,
        inner_radius=INNER_RADIUS,
        outer_radius=OUTER_RADIUS,
    )

    res = run_fastpaco_original(cube, angles, psf, pixscale, phi0s, cpu=CPU)
    res["cube_shape"] = tuple(cube.shape)
    res["pixscale"] = float(pixscale)
    res["phi0s"] = phi0s
    results_original[ds_name] = res

    print(f"Tiempo: {res['time_s']:.2f}s | pixeles: {res['n_pixels']} | SNR max: {res['snr_max']:.2f}")
    print(f"SNR mean: {res['snr_mean']:.3f} | std: {res['snr_std']:.3f}")
    print(f"SNR>3: {res['count_snr_gt_3']} | SNR>5: {res['count_snr_gt_5']} | SNR>7: {res['count_snr_gt_7']}")

    rows.append({
        "Dataset": ds_name,
        "Tiempo original (s)": res["time_s"],
        "Pixeles": res["n_pixels"],
        "SNR max original": res["snr_max"],
        "SNR mean original": res["snr_mean"],
        "SNR>3": res["count_snr_gt_3"],
        "SNR>5": res["count_snr_gt_5"],
        "SNR>7": res["count_snr_gt_7"],
    })

df_original = pd.DataFrame(rows)
display(df_original)


In [ ]:
# Comparacion contra referencia TESISPRIMERBATCH
compare_rows = []

for ds_name, res in results_original.items():
    ref = REFERENCE_BATCH.get(ds_name)
    if ref is None:
        continue

    dmax = float(res["snr_max"] - ref["snr_max"])
    dmean = float(res["snr_mean"] - ref["snr_mean"])
    px_match = int(res["n_pixels"]) == int(ref["pixels"])
    robust_ok = (abs(dmax) <= SNR_MAX_TOL) and (abs(dmean) <= SNR_MEAN_TOL) and px_match

    compare_rows.append({
        "Dataset": ds_name,
        "Pixeles original": res["n_pixels"],
        "Pixeles referencia": ref["pixels"],
        "SNR max original": res["snr_max"],
        "SNR max referencia": ref["snr_max"],
        "Delta SNR max": dmax,
        "SNR mean original": res["snr_mean"],
        "SNR mean referencia": ref["snr_mean"],
        "Delta SNR mean": dmean,
        "Robustez OK": "SI" if robust_ok else "REVISAR",
    })

df_compare = pd.DataFrame(compare_rows)
display(df_compare)

if len(df_compare):
    n_ok = int(np.sum(df_compare["Robustez OK"] == "SI"))
    print(f"Datasets OK: {n_ok}/{len(df_compare)}")
else:
    print("No hay referencias para comparar. Revisa REFERENCE_BATCH.")


In [ ]:
# Visualizaciones (tabla + graficos, estilo TESISPRIMERBATCH)
if len(df_original) == 0:
    print("No hay resultados para visualizar")
else:
    display(df_original.style.format({
        "Tiempo original (s)": "{:.2f}",
        "SNR max original": "{:.2f}",
        "SNR mean original": "{:.3f}",
    }).set_caption("Resultados PACO original"))

    # 1) Barras de tiempo y SNR max por dataset
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(df_original["Dataset"], df_original["Tiempo original (s)"], color="#1f77b4")
    axes[0].set_title("Tiempo por dataset (PACO original)")
    axes[0].set_ylabel("Segundos")
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(df_original["Dataset"], df_original["SNR max original"], color="#ff7f0e")
    axes[1].set_title("SNR max por dataset")
    axes[1].set_ylabel("SNR max")
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # 2) Conteo de detecciones por umbral
    idx = np.arange(len(df_original))
    w = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(idx - w, df_original["SNR>3"], width=w, label="SNR > 3")
    ax.bar(idx, df_original["SNR>5"], width=w, label="SNR > 5")
    ax.bar(idx + w, df_original["SNR>7"], width=w, label="SNR > 7")
    ax.set_xticks(idx)
    ax.set_xticklabels(df_original["Dataset"].tolist())
    ax.set_title("Conteo de detecciones por umbral")
    ax.set_ylabel("Cantidad")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    # 3) Histogramas de SNR para cada dataset
    n_ds = len(results_original)
    fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 4), squeeze=False)
    for i, (ds_name, res) in enumerate(results_original.items()):
        ax = axes[0, i]
        snr = np.asarray(res["snr"])
        ax.hist(snr, bins=60, color="#2ca02c", alpha=0.8)
        ax.set_title(f"Histograma SNR - {ds_name}")
        ax.set_xlabel("SNR")
        ax.set_ylabel("Frecuencia")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 4) Mapa SNR sobre coordenadas evaluadas (phi0s)
    n_ds = len(results_original)
    fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 5), squeeze=False)
    for i, (ds_name, res) in enumerate(results_original.items()):
        ax = axes[0, i]
        h, w_img = int(res["cube_shape"][1]), int(res["cube_shape"][2])
        snr_map = np.full((h, w_img), np.nan, dtype=float)
        phi0s = np.asarray(res["phi0s"])
        snr = np.asarray(res["snr"])
        snr_map[phi0s[:, 1], phi0s[:, 0]] = snr

        im = ax.imshow(snr_map, origin="lower", cmap="inferno")
        ax.set_title(f"Mapa SNR (anillo) - {ds_name}")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

if 'df_compare' in globals() and len(df_compare):
    display(df_compare.style.format({
        "SNR max original": "{:.2f}",
        "SNR max referencia": "{:.2f}",
        "Delta SNR max": "{:+.2f}",
        "SNR mean original": "{:.3f}",
        "SNR mean referencia": "{:.3f}",
        "Delta SNR mean": "{:+.3f}",
    }).set_caption("Comparacion PACO original vs TESISPRIMERBATCH"))

In [ ]:
# Exportar tablas
out_dir = Path('/content')
df_original.to_csv(out_dir / 'paco_original_resultados.csv', index=False)
if 'df_compare' in globals() and len(df_compare):
    df_compare.to_csv(out_dir / 'paco_original_vs_primerbatch.csv', index=False)
print('Exportado en /content:')
print('- paco_original_resultados.csv')
print('- paco_original_vs_primerbatch.csv (si hubo referencia)')
